## 1. Imports

In [1]:
import pandas as pd
import joblib

from pathlib import Path

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    mean_absolute_error,
    root_mean_squared_error,
    r2_score,
)
from sklearn.model_selection import (
    train_test_split,
    GridSearchCV,
)

from sklearn.pipeline import Pipeline

from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler,
)

from sklearn.ensemble import RandomForestRegressor

## 2. Load data

In [2]:
DF_PATH = Path("../data/processed/feature_engineered_df.parquet")
df = pd.read_parquet(DF_PATH)

df.head()

,property_type,room_type,accommodates,bathrooms,bed_type,cancellation_policy,cleaning_fee,city,host_has_profile_pic,host_identity_verified,...,has_wifi,has_kitchen,has_heating,description_length,description_word_count,does_host_respond,accommodates_per_bedroom,beds_per_bedroom,bathrooms_per_bedroom,log_price
0,Apartment,Entire home/apt,3,1.0,Real Bed,strict,1,NYC,t,t,...,1,1,1,211,31,0,3.000000,1.0,1.000000,5.010635
1,Apartment,Entire home/apt,7,1.0,Real Bed,strict,1,NYC,t,f,...,1,1,1,1000,172,1,2.333333,1.0,0.333333,5.129899
2,Apartment,Entire home/apt,5,1.0,Real Bed,moderate,1,NYC,t,t,...,1,1,1,1000,172,1,5.000000,3.0,1.000000,4.976734
3,House,Entire home/apt,4,1.0,Real Bed,flexible,1,SF,t,t,...,1,1,1,468,78,0,2.000000,1.0,0.500000,6.620073
4,Apartment,Entire home/apt,2,1.0,Real Bed,moderate,1,DC,t,t,...,1,1,1,699,120,1,0.000000,0.0,0.000000,4.744932


## 3. Separate features and target

In [3]:
X = df.drop("log_price", axis=1)
y = df["log_price"]

## 4. Train test split

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    shuffle=True
)

## 5. Identify column types

In [5]:
categorical_columns = X_train.select_dtypes(include=["object", "string"]).columns.to_list()
numerical_columns = X_train.select_dtypes(include=["number", "bool"]).columns.to_list()

print(f"Numerical columns: ({len(numerical_columns)})")
print()
print(f"Categorical columns: ({len(categorical_columns)})")

Numerical columns: (24)

Categorical columns: (10)


## 6. Build preprocessing pipelines

In [6]:
numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore")),
    ]
)

## 7. ColumnTransformer

In [7]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numerical_columns),
        ("cat", categorical_pipeline, categorical_columns),
    ]
)

## 8. Create random forest model

In [8]:
random_forest = RandomForestRegressor(
    random_state=42, 
    n_jobs=-1,
)

## 9. Create random forest pipeline

In [9]:
rand_for_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", random_forest),
    ]
)

## 10. Define parameter grid

In [10]:
param_grid = {
    'model__n_estimators': [100, 200],
    'model__max_depth': [10, 20],
    'model__min_samples_split': [2, 10],
    'model__min_samples_leaf': [1, 2],
}

## 11. Fit the model

In [11]:
grid = GridSearchCV(rand_for_pipeline, param_grid, cv=3, scoring="r2", verbose=2)
model_grid = grid.fit(X_train, y_train)

print(f"Best hyperparameters: {model_grid.best_params_}")
print(f"Best CV R²: {model_grid.best_score_:.4f}")

Fitting 3 folds for each of 16 candidates, totalling 48 fits
[CV] END model__max_depth=10, model__min_samples_leaf=1, model__min_samples_split=2, model__n_estimators=100; total time=  11.2s
[CV] END model__max_depth=10, model__min_samples_leaf=1, model__min_samples_split=2, model__n_estimators=100; total time=  10.8s
[CV] END model__max_depth=10, model__min_samples_leaf=1, model__min_samples_split=2, model__n_estimators=100; total time=  36.7s
[CV] END model__max_depth=10, model__min_samples_leaf=1, model__min_samples_split=2, model__n_estimators=200; total time=  45.5s
[CV] END model__max_depth=10, model__min_samples_leaf=1, model__min_samples_split=2, model__n_estimators=200; total time=  52.6s
[CV] END model__max_depth=10, model__min_samples_leaf=1, model__min_samples_split=2, model__n_estimators=200; total time=  49.3s
[CV] END model__max_depth=10, model__min_samples_leaf=1, model__min_samples_split=10, model__n_estimators=100; total time=  12.4s
[CV] END model__max_depth=10, model

## 12. Calculate metrics

In [12]:
y_pred = model_grid.predict(X_test)

metrics_df = pd.DataFrame([{
    "Model Name": "Random Forest (Tuned)",
    "CV Mean R² score": model_grid.best_score_,
    "CV Std": model_grid.cv_results_["std_test_score"][model_grid.best_index_],
    "Test R² score": r2_score(y_test, y_pred),
    "MAE": mean_absolute_error(y_test, y_pred),
    "RMSE": root_mean_squared_error(y_test, y_pred),
}]).round(2)

# look at the metrics result
metrics_df

,Model Name,CV Mean R² score,CV Std,Test R² score,MAE,RMSE
0,Random Forest (Tuned),0.7,0.0,0.71,0.28,0.39


## 13. Save the metrics result and the cv results

In [13]:
# metrics first
METRICS_DF_PATH = Path("../data/processed/best_model_metrics.csv")
metrics_df.to_csv(METRICS_DF_PATH, index=False)

# cv results
CV_RESULTS_PATH = Path("../data/processed/cv_results.csv")
cv_results_df = pd.DataFrame(model_grid.cv_results_)
cv_results_df.to_csv(CV_RESULTS_PATH, index=False)

## 14. Save the best model

In [14]:
BEST_MODEL_PATH = Path("../models/random_forest_tuned.pkl")
joblib.dump(model_grid.best_estimator_, BEST_MODEL_PATH)

['..\\models\\random_forest_tuned.pkl']